# 문제 6

[Kaggle 형] train_prob.csv로 failure 예측하는 모델을 만들고, 

test_prob.csv에 대한 failure가 1일 확률 예측하여 다음과 같은 형식의 answer6.csv를 만들어라. 

측정 지표는 AUC(area under of ROC curve)이다. id 는 테스트 케이스의 id 이고, failure에는 failure가 1이 될 확률이다.

id,failure

16115, 0.1

16116, 0.2


**강사: 멀티캠퍼스 강선구(sunku0316.kang@multicampus.com, sun9sun9@gmail.com)**

In [1]:
# 실행 환경 확인

import pandas as pd
import numpy as np
import sklearn
import scipy
import statsmodels
import mlxtend # !pip install --upgrade mlxtend
import sys
import xgboost as xgb # !pip install --upgrade xgboost

print(sys.version)
for i in [pd, np, sklearn, scipy, mlxtend, statsmodels, xgb]:
    print(i.__name__, i.__version__)

3.7.4 (tags/v3.7.4:e09359112e, Jul  8 2019, 20:34:20) [MSC v.1916 64 bit (AMD64)]
pandas 0.25.1
numpy 1.18.5
sklearn 0.21.3
scipy 1.5.2
mlxtend 0.15.0.0
statsmodels 0.11.1
xgboost 0.80


# Kaggle형 풀이 단계

Step 0: Kaggle용 데이터셋을 만든다.

Step 1: 검증 방법을 정하고, 검증 루틴을 만듭니다.

Step 2: Baseline 모델을 만듭니다

Step 3: 모델 선택 루틴을 만듭니다.

|id|failure|
|----|----|
|16115| 0.1|
|16116| 0.2|

....	

Step 4: 모델 개선 작업을 합니다.

## Step 0: Kaggle용 데이터셋을 만든다.

In [5]:
# 데이터를 식별할 만한 변수(고윳값)가 있으면 인덱스로 사용합니다.
# 여기서는 id를 인덱스로 지정해볼만 합니다.
# 이 과정은 필수는 아닙니다.

df_train = pd.read_csv('train_prob.csv', index_col = ['id'])
df_test = pd.read_csv('test_prob.csv', index_col = ['id'])
s_kaggle_ans = pd.read_csv('test_prob_ans.csv', index_col = ['id'])['failure']

In [6]:
# from 문제 1
df_train = df_train.assign(
    na_1 = lambda x: x['measurement_3'].isna(),
    na_2 = lambda x: x['measurement_5'].isna(),
)
df_test = df_test.assign(
    na_1 = lambda x: x['measurement_3'].isna(),
    na_2 = lambda x: x['measurement_5'].isna(),
)

In [7]:
df_train['product_code'].value_counts()

C    5765
E    5343
B    5250
A    5100
Name: product_code, dtype: int64

In [8]:
df_test['product_code'].value_counts()

D    5112
Name: product_code, dtype: int64

In [10]:
# 방법 2: groupby ~ apply ~ fit_transform
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

X_imp = ['measurement_{}'.format(i) for i in range(3, 10)] + ['measurement_17']
imp = IterativeImputer(
    estimator = LinearRegression(fit_intercept = True), random_state=123
)

df_train[X_imp] = df_train.groupby('product_code')[X_imp].apply(
    lambda x: pd.DataFrame(imp.fit_transform(x), index = x.index, columns = X_imp)
)
df_test[X_imp] = df_test.groupby('product_code')[X_imp].apply(
    lambda x: pd.DataFrame(imp.fit_transform(x), index = x.index, columns = X_imp)
)

In [11]:
X_mean = ['measurement_{}'.format(i) for i in range(10, 17)]
df_train[X_mean] = df_train.groupby('product_code')[X_mean].transform(
    lambda x: x.fillna(x.mean())
)
df_test[X_mean] = df_test.groupby('product_code')[X_mean].transform(
    lambda x: x.fillna(x.mean())
)

In [14]:
m = pd.concat(
    [df_train['loading'], df_test['loading']]
).mean()
df_train['loading'] = df_train['loading'].fillna(m)
df_test['loading'] = df_test['loading'].fillna(m)

In [ ]:
# SFS: LR, STD('loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17') 'na_1'
# LDA: predictor, transformer
# PCA: (n_components = 7 measurement_0 ~ 17) + loading
# loading -> log 변환 -> loading_log 
# GS: {'n_estimators': 15, 'max_depth': 7, 'min_samples_split': 512}, 0.5745226991354744

## Step1: 검증 방법을 정하고, 검증 루틴을 만듭니다.

## Step2: Baseline 모델을 만듭니다.

## Step3: 모델 선택 루틴을 만듭니다.

## Step4: 모델 개선을 해봅니다.

In [ ]:
# Baseline 튜닝: C
# std: ['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17'] + pt: ['na_1'] -> LR

### LR2

LR + PCA - measurement_0~17 -> STD -> PCA(n_components = 7) + loading -> STD,  PT: na_1, na_2

### LR3

Baseline + loading_log

### LDA

LDA + Baseline

### RF

- ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(18)]
- RF: RandomForestClassifier: n_estimators=100, max_depth=6, min_samples_split=512, random_state=123
- max_features를 사용해 튜닝

### RF2: RandomForestClassifier + LinearDiscriminantAnalysis

### XGB
- ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(18)]
- n_estimators=300, learning_rate=0.01, colsample_bytree=0.85
- subsample을 튜닝

### Voting

- ('baseline', clf_lr), # LR + SFS
- ('lr2', clf_lr2), # LR.2: LR + feature PCA
- ('lda', clf_lda), # LDA
- ('rf2', clf_rf2), # RF + LDA

## Stacking

다른 앙상블 기법인 Stacking을 보여드립니다.


참고용입니다. 지금까지 했던 데이터 처리와 머신러닝 기법을 복습해보기 위해 준비했습니다.